# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [4]:
# Load environment variables
load_dotenv()

True

### VectorDB Instance

In [5]:
# Instantiate your ChromaDB Client
# PersistentClient writes to disk so the collection survives the notebook
# and can be reopened from the agent in Part 02
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [6]:
# Pick one embedding function
# Using OpenAI embeddings (the same one must be used when loading it in Part 02)
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY")
)

In [8]:
# Create a collection
collection = chroma_client.create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

### Add documents

In [9]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

### Verify the collection

Confirm every game was indexed and that semantic search returns sensible results.

In [10]:
# How many documents were indexed?
print(f"Documents in collection: {collection.count()}")

# Sanity-check a semantic query
results = collection.query(
    query_texts=["realistic racing simulator"],
    n_results=3,
    include=['documents', 'metadatas', 'distances']
)

for doc, meta, distance in zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
):
    print(f"\n(distance={distance:.3f}) {meta['Name']} [{meta['Platform']}, {meta['YearOfRelease']}]")
    print(f"    {doc}")

Documents in collection: 15

(distance=0.124) Gran Turismo [PlayStation 1, 1997]
    [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.

(distance=0.134) Gran Turismo 5 [PlayStation 3, 2010]
    [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.

(distance=0.198) Mario Kart 8 Deluxe [Nintendo Switch, 2017]
    [Nintendo Switch] Mario Kart 8 Deluxe (2017) - An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.
